# Master TIDE - Session 4

1. **Web scraping et parsing HTML**
3. **API**

### 1. Web Scraping

- Extraction d'informations d'un site web.
- A utiliser en l'absence de données ouvertes ou d'API.
- Technique fragile car le site web peut changer du jour au lendemain.
- Problématique juridique...

**Avec requests**

Doc :
- requests : https://requests.readthedocs.io/en/master/

In [ ]:
import requests

r = requests.get('https://kim.fspot.org/cours/page2.html')
r.status_code

In [ ]:
# content
type(r.content)

In [ ]:
# type
type(r.content)

In [ ]:
# str en précisant un encodage
content = r.content.decode('utf-8')
type(content)

In [ ]:
# BeautifulSoup

from bs4 import BeautifulSoup

soup = BeautifulSoup(content, "html.parser")
soup

On utilise différentes méthodes :
    
- `find(tag, attrs)` : trouve le premier tag avec les attributs spécifiés
- `findAll(tag, attrs)` : trouve tous les tags avec les attributs spécifiés
- `select(css)` : trouve tous les tags avec les css spécifiées
- etc.

La technique consiste par exemple à alimenter un dictionnaires avec les valeurs trouvées :
- soit en utilisant `tag.attrs['attr']` pour collecter la valeur attr du tag
- soit en utilisant `tag.text` pour collecter la valeur `text`
- on peut aussi utiliser d'autres méthodes comme `select()` qui joue sur la CSS 
- éventuellement en recherchant dans un nouveau tag à l'intérieur d'un tag donné

In [ ]:
# exploration du HTML

div_tag = soup.find('div', attrs={'class': 'pricing-table'})
div_tag

Méthode :
 1. On collecte les informations d'un  tag pour un seul enregistrement sous la forme d'un dictionaire.
 2. On généralise en bouclant sur tous les tags de même nature et on rassemble les dictionnaires dans une liste.
 3. On transforme la liste de dictionaires en DataFrame.

In [ ]:
# Transformation en DataFrame
import pandas as pd
df = pd.DataFrame(rows)
df

#### Inconvénients du web scraping:
- plutôt lent (car on parse potentiellement beaucoup de HTML inutile)
- ne donne pas les résultats attendus si une partie du contenu est intégré dynamiquement à la page via javascript
- un changement dans l'architecture du html ou du css (e.g: refonte du design du site) oblige à réécrire entièrement le programme

### API

Exemple: Deezer

Artiste : https://www.deezer.com/fr/artist/3037

Récupérer le nombre de fans d'un artiste avec requests (cherchez le tag div avec l'id `"naboo_artist_social_small"`).

In [ ]:
import requests
from bs4 import BeautifulSoup
# request
artist = 3037
response = requests.get(f'https://www.deezer.com/fr/artist/{artist}')
len(response.content)

Récupérer le nombre de fans d'un artiste avec l'API :

Doc:
- https://pypi.org/project/deezer-python/

Installation :
- *pip install deezer-python*

Le terme "API" est très générique et peut désigner bien des choses, mais dans le jargon on l'utilise souvent pour désigner un service web qui renvoie non pas:
> des pages web au format HTML (destinées à être lues par un humain dans son navigateur)

mais:
> des données au format JSON (destinées à être traitées par un programme)

![img](https://miro.medium.com/max/4238/1*OcmVkcsM5BWRHrg8GC17iw.png)

Puisque les API sont dédiées à l'usage via des programmes, elles disposent en général d'une bonne documentation, et sont fiables et stables dans le temps. Tandis que sur des pages web HTML classiques, le design peut par exemple changer du jour au lendemain et rendre votre programme BeautifulSoup obsolète.

In [ ]:
# API JSON
response = requests.get(f'https://api.deezer.com/artist/{artist}')
response.content

In [ ]:
# nb_fan
data = response.json()
data['nb_fan']

In [ ]:
# picture
from IPython.display import Image
url = data['picture']
r = requests.get(url)
Image(data=r.content)

#### Avantages d'une API
- renvoie du format JSON, facile et rapide à traiter
- renvoie un format stable et documenté (voire versionné)
- exemple : https://developer.twitter.com/en/docs/twitter-api/api-reference-index
- la documentation indique comment interagir avec l'API:
    - quelle url
    - quelle méthode http (GET, POST, ...)
    - quels paramètres
    - ...
→ idéal pour les développeurs

### Quel intérêt pour le fournisseur d'API ?

En général il met en place des quotas de requêtes ou d'autres limitations afin de proposer un service payant qui dispose de possibilités avancées / d'un meilleur support / etc.

C'est pourquoi de nombreux services nécessitent de se connecter avec son compte client pour utiliser une API (e.g. https://openweathermap.org/api)

#### **Basic Auth**

Exemple: accéder à https://kim.fspot.org/private/

Pour y accéder il est nécessaire d'utiliser les credentials suivant:
- login: admin
- password: secret

Si on ne les passe pas (ou si on ne passe pas les bons), on a une erreur 401 (= unauthorized).

In [ ]:
# sans login/password
res = requests.get('https://kim.fspot.org/private')
res

In [ ]:
# avec login password
res = requests.get('https://kim.fspot.org/private', auth=('admin', 'secret'))
res

In [ ]:
# contenu
res.content.decode('utf-8')

In [ ]:
# YAML pour masquer auth crendentials

#### Auth par token

Exemple sur openweathermap :
- documentation: https://openweathermap.org/appid
- mes tokens: https://home.openweathermap.org/api_keys

In [ ]:
# requête avec un token
token = ''

Avantage des tokens:
- évite que les requêtes HTTP contiennent le mot de passe - à la place elles contiennent un token
- si je me fais "voler" un token, je peux le supprimer de mon compte

In [ ]:
# requête avec un token
url = f'http://api.openweathermap.org/data/2.5/weather?APPID={token}&q=Paris'
res = requests.get(url)
meteo = res.json()
meteo

In [ ]:
# type
type(meteo)

In [ ]:
# extractions
{'city': meteo['name'],
'country': meteo['sys']['country'],
'date': meteo['dt'],
'temp': meteo['main']['temp'] - 273.15,}

In [ ]:
# extractions
import time

{'city': meteo['name'],
'country': meteo['sys']['country'],
'date': time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(meteo['dt'])),
'temp': meteo['main']['temp'] - 273.15,}